# EDA Survei Kos Toba (928 Baris)

Notebook ini menyelesaikan seluruh tugas Issue #15 pada dataset survei terbaru (`data/raw/Survey_Kos.csv`, 928 baris):
- distribusi harga bulanan normalisasi (`harga_bulanan_norm`);
- harga berdasarkan 16 kecamatan dan kategori penghuni;
- distribusi luas kamar (`luas_kamar_m2`) dan kategori lantai (`1`, `2`, `3`, `4 - 5`);
- hubungan fasilitas kamar, kamar mandi, dan sistem keamanan (`CCTV`, `Penjaga/Satpam`, `Penjaga & CCTV`) dengan harga;
- hubungan 7 variabel jarak dalam satuan meter (`_m` dan `klasifikasi_jarak_*`) dengan harga;
- evaluasi kualitas data 4 dimensi, missing value, outlier IQR, dan status data;
- minimal 5 temuan utama tertulis beserta interpretasi bisnis untuk setiap grafik.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATASET_VERSION = 'survey_clean_v1'
SEED = 42
project = next(path for path in [Path.cwd(), *Path.cwd().parents]
               if (path / 'data/raw/Survey_Kos.csv').exists())
raw_path = project / 'data/raw/Survey_Kos.csv'
staging_path = project / 'data/staging/survey_clean_v1.csv'
warehouse_path = project / 'data/tobakost_survey_warehouse.db'
figure_dir = project / 'reports/figures/issue-15'
figure_dir.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

raw = pd.read_csv(raw_path)
df = pd.read_csv(staging_path)
analysis = df.loc[df['accepted_for_analysis'].eq(1)].copy()

with sqlite3.connect(warehouse_path) as conn:
    warehouse_count = conn.execute('SELECT COUNT(*) FROM fact_sewa_kos').fetchone()[0]
    model_ready_count = conn.execute('SELECT COUNT(*) FROM fact_sewa_kos WHERE model_ready = 1').fetchone()[0]

print({'dataset_version': DATASET_VERSION, 'seed': SEED, 'raw_rows': len(raw),
       'staging_rows': len(df), 'analysis_rows': len(analysis),
       'warehouse_fact_rows': warehouse_count, 'model_ready_rows': model_ready_count})

## Ringkasan status kualitas data

In [2]:
quality_summary = pd.DataFrame({
    'Status': ['Exact duplicate', 'Duplikasi logis', 'Kandidat outlier IQR',
               'Lolos analisis (Warehouse Fact)', 'Model ready'],
    'Jumlah': [int(df['exact_duplicate'].sum()), int(df['duplicate_logis'].sum()),
               int(df['price_anomaly_iqr'].sum()), int(df['accepted_for_analysis'].sum()),
               int(df['model_ready'].sum())]
})
quality_summary

## 1. Distribusi harga bulanan normalisasi

In [3]:
prices = pd.to_numeric(analysis['harga_bulanan_norm'], errors='coerce').dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(prices, bins=24, color='#2563eb', edgecolor='white', alpha=0.85)
axes[0].axvline(prices.median(), color='#dc2626', linestyle='--',
                label=f'Median Rp{prices.median():,.0f}')
axes[0].set(title='Histogram Harga Setara Bulanan (n=923)', xlabel='Tarif bulanan (Rp)',
            ylabel='Jumlah unit')
axes[0].ticklabel_format(style='plain', axis='x')
axes[0].legend()
axes[1].boxplot(prices, vert=True, patch_artist=True,
                boxprops={'facecolor':'#dbeafe'})
axes[1].set(title='Boxplot Harga Setara Bulanan (n=923)', ylabel='Tarif bulanan (Rp)')
axes[1].ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'01_distribusi_harga.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print({'mean': round(prices.mean(), 2), 'median': prices.median(), 'q1': prices.quantile(.25), 'q3': prices.quantile(.75)})

**Interpretasi (Temuan 1):** Pada 923 baris unit kos yang lolos validasi analisis, median harga setara bulanan adalah **Rp700.000** (rata-rata **Rp693.929**), dengan rentang interkuartil (IQR) yang sangat terkonsentrasi antara **Q1 = Rp575.000** dan **Q3 = Rp800.000** (rentang total Rp100.000 hingga Rp1.550.000/bulan). Penggunaan median dan IQR memberikan tolok ukur harga pasar yang tangguh terhadap unit kos eksekutif bertarif di atas Rp1,14 juta/bulan.

## 2. Harga berdasarkan kecamatan dan kategori penghuni

In [4]:
kec = analysis.groupby('kecamatan')['harga_bulanan_norm'].agg(['count','median','mean']).sort_values('median')
category = analysis.groupby('kategori_penghuni')['harga_bulanan_norm'].agg(['count','median','mean']).sort_values('median')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(kec.index, kec['median'], color='#2563eb')
axes[0].set(title='Median Harga per Kecamatan (16 Kecamatan)', xlabel='Kecamatan', ylabel='Median Rp/bulan')
axes[0].tick_params(axis='x', rotation=45)
axes[0].ticklabel_format(style='plain', axis='y')
axes[1].bar(category.index, category['median'], color='#14b8a6')
axes[1].set(title='Median Harga per Kategori Penghuni', xlabel='Kategori', ylabel='Median Rp/bulan')
axes[1].tick_params(axis='x', rotation=25)
axes[1].ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'02_harga_kecamatan_kategori.png', dpi=150, bbox_inches='tight')
plt.close(fig)
display(kec)
display(category)

**Interpretasi (Temuan 2):** Terdapat segmentasi geografis yang jelas lintas 16 kecamatan di Kabupaten Toba. Empat kecamatan pusat pendidikan, pemerintahan, dan pariwisata mempunyai median tertinggi, yaitu **Laguboti (Rp725.000, n=254)**, **Porsea (Rp725.000, n=104)**, **Ajibata (Rp725.000, n=56)**, dan **Balige (Rp700.000, n=355)**, sedangkan kecamatan pinggiran seperti **Pintu Pohan Meranti (Rp352.084)**, **Habinsaran (Rp425.000)**, dan **Sigumpar (Rp475.000)** memiliki tarif jauh lebih rendah. Berdasarkan kategori penghuni, kos **Putri (median Rp725.000, n=264)** dan **Campur (median Rp725.000, n=403)** memiliki tarif median Rp150.000 lebih tinggi dibandingkan kos **Putra (median Rp575.000, n=256)**.

## 3. Distribusi luas kamar dan kategori lantai

In [5]:
areas = pd.to_numeric(analysis['luas_kamar_m2'], errors='coerce').dropna()
floor = analysis.groupby('kategori_lantai')['harga_bulanan_norm'].agg(['count','median','mean'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(areas, bins=20, color='#8b5cf6', edgecolor='white')
axes[0].axvline(areas.median(), color='#dc2626', linestyle='--',
                label=f'Median {areas.median():.1f} m²')
axes[0].set(title='Distribusi Luas Kamar (m²)', xlabel='Luas kamar (m²)', ylabel='Jumlah unit')
axes[0].legend()
axes[1].bar(floor.index.astype(str), floor['median'], color='#06b6d4')
axes[1].set(title='Median Harga per Kategori Lantai', xlabel='Kategori Lantai', ylabel='Median Rp/bulan')
axes[1].ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'03_luas_dan_lantai.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print({'median_luas': areas.median(), 'q1_luas': areas.quantile(.25), 'q3_luas': areas.quantile(.75)})
display(floor)

**Interpretasi (Temuan 3):** Setelah normalisasi ekspresi perkalian dimensi (`4x4`, `3,5x5`) dan koreksi typo angka ekstrem (`2000 -> 200` m²), luas kamar kos di Kabupaten Toba memiliki **median 12,0 m²** (setara kamar $3\times 4$ m, dengan Q1 = 10,0 m² dan Q3 = 13,0 m²). Berdasarkan kategori lantai, **Lantai 2 (n=290) memiliki median tarif Rp750.000**, lebih tinggi Rp100.000 dibandingkan **Lantai 1 (n=630, median Rp650.000)** karena bangunan kos 2 lantai umumnya merupakan bangunan baru dengan fasilitas kamar mandi dalam. Kategori lantai `3` (n=2) dan `4 - 5` (n=1) memiliki ukuran sampel sangat kecil.

## 4. Hubungan fasilitas dan keamanan dengan harga

In [6]:
wifi_yes = analysis['is_wifi'].eq(1)
km_yes = analysis['km_dalam'].eq(1)
facility = pd.DataFrame({
    'Kelompok': ['AC', 'AC', 'Wi-Fi', 'Wi-Fi', 'Kamar Mandi', 'Kamar Mandi', 'Keamanan', 'Keamanan'],
    'Fasilitas': ['Ada AC', 'Tanpa AC', 'Ada Wi-Fi', 'Tanpa Wi-Fi', 'KM Dalam', 'KM Luar', 'Ada Keamanan (CCTV/Penjaga)', 'Tanpa Keamanan'],
    'Median': [
        analysis.loc[analysis['ac'].eq(1), 'harga_bulanan_norm'].median(),
        analysis.loc[analysis['ac'].eq(0), 'harga_bulanan_norm'].median(),
        analysis.loc[wifi_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[~wifi_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[km_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[~km_yes, 'harga_bulanan_norm'].median(),
        analysis.loc[analysis['keamanan_kos'].ne('Tidak Ada'), 'harga_bulanan_norm'].median(),
        analysis.loc[analysis['keamanan_kos'].eq('Tidak Ada'), 'harga_bulanan_norm'].median(),
    ],
    'Sampel': [
        int(analysis['ac'].eq(1).sum()), int(analysis['ac'].eq(0).sum()),
        int(wifi_yes.sum()), int((~wifi_yes).sum()),
        int(km_yes.sum()), int((~km_yes).sum()),
        int(analysis['keamanan_kos'].ne('Tidak Ada').sum()), int(analysis['keamanan_kos'].eq('Tidak Ada').sum()),
    ]
})
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(facility['Fasilitas'], facility['Median'], color=['#10b981','#94a3b8']*4)
ax.set(title='Median Tarif Berdasarkan Fasilitas & Keamanan', xlabel='Fasilitas', ylabel='Median Rp/bulan')
ax.tick_params(axis='x', rotation=20)
ax.ticklabel_format(style='plain', axis='y')
fig.tight_layout()
fig.savefig(figure_dir/'04_fasilitas_vs_harga.png', dpi=150, bbox_inches='tight')
plt.close(fig)
display(facility)
display(analysis.groupby('keamanan_kos')['harga_bulanan_norm'].agg(['count','median','mean']).sort_values('median'))

**Interpretasi (Temuan 4):** Spesifikasi fasilitas memberikan selisih tarif median yang sangat signifikan:
- **AC:** Unit ber-AC (n=42) memiliki median **Rp1.200.000/bulan** vs Tanpa AC (n=881) sebesar **Rp675.000/bulan** (+Rp525.000).
- **Wi-Fi:** Unit dengan Wi-Fi (n=667) memiliki median **Rp750.000/bulan** vs Tanpa Wi-Fi (n=256) sebesar **Rp550.000/bulan** (+Rp200.000).
- **Tipe Kamar Mandi:** Kamar mandi `dalam` (n=863) memiliki median **Rp700.000/bulan** vs `luar` (n=60) sebesar **Rp500.000/bulan** (+Rp200.000).
- **Keamanan Kos:** Tarif median meningkat berjenjang sesuai tingkat keamanan: **Tidak Ada (Rp625.000, n=459)** $\rightarrow$ **CCTV (Rp650.000, n=176)** $\rightarrow$ **Penjaga/Satpam (Rp750.000, n=215)** $\rightarrow$ **Penjaga & CCTV (Rp850.000, n=73)**.

## 5. Hubungan jarak fasilitas (meter) dengan harga

In [7]:
distance_columns = [column for column in analysis.columns if column.startswith('jarak_') and column.endswith('_m')]
distance_corr = {}
for column in distance_columns:
    pair = pd.DataFrame({'harga': pd.to_numeric(analysis['harga_bulanan_norm'], errors='coerce'),
                         'jarak': pd.to_numeric(analysis[column], errors='coerce')}).dropna()
    distance_corr[column] = pair.corr(method='spearman').iloc[0,1] if len(pair) >= 2 else 0
corr = pd.Series(distance_corr).sort_values()
labels = [name.replace('jarak_','').replace('_m','').replace('_',' ').title() for name in corr.index]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels, corr.values, color='#6366f1')
ax.axvline(0, color='#0f172a', linewidth=.8)
ax.set(title='Korelasi Spearman Jarak Fasilitas (Meter) vs Harga Bulanan', xlabel='Koefisien korelasi Spearman')
fig.tight_layout()
fig.savefig(figure_dir/'05_jarak_vs_harga.png', dpi=150, bbox_inches='tight')
plt.close(fig)
display(corr.rename(index=dict(zip(corr.index, labels))))
display(analysis.groupby('klasifikasi_jarak_kampus_sekolah_tempatkerja')['harga_bulanan_norm'].agg(['count','median','mean']))

**Interpretasi (Temuan 5):** Dalam satuan meter (`_m`), **jarak masuk dari jalan raya utama (`jarak_jalan_raya_m`) memiliki korelasi negatif terkuat terhadap harga ($\rho = -0,3975$)**, disusul jarak ke kampus/sekolah/tempat kerja ($\rho = -0,0652$). Artinya, semakin dekat lokasi kos ke jalan raya utama dan pusat aktivitas (`Sangat Dekat <= 500 m` memiliki median **Rp700.000**, sedangkan `Jauh > 3000 m` turun ke **Rp570.834**), semakin tinggi tarif sewanya.

## 6. Kualitas data, outlier IQR, dan status data

In [8]:
q1 = prices.quantile(.25)
q3 = prices.quantile(.75)
lower = q1 - 1.5 * (q3-q1)
upper = q3 + 1.5 * (q3-q1)
status_counts = pd.Series({
    'Total Staging': len(df),
    'Lolos Analisis': int(df['accepted_for_analysis'].sum()),
    'Model Ready': int(df['model_ready'].sum()),
    'Duplikasi Logis': int(df['duplicate_logis'].sum()),
    'Outlier IQR': int(df['price_anomaly_iqr'].sum()),
    'Ditolak/Non-Aktif': int((df['accepted_for_analysis'] == 0).sum()),
})
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].boxplot(prices, vert=True, patch_artist=True, boxprops={'facecolor':'#dbeafe'})
axes[0].axhline(lower, color='#dc2626', linestyle='--', label=f'Batas bawah Rp{lower:,.0f}')
axes[0].axhline(upper, color='#dc2626', linestyle='--', label=f'Batas atas Rp{upper:,.0f}')
axes[0].set(title='Outlier Harga Bulanan (Metode IQR)', ylabel='Rp/bulan')
axes[0].ticklabel_format(style='plain', axis='y')
axes[0].legend()
axes[1].barh(status_counts.index, status_counts.values, color='#f97316')
axes[1].set(title='Ringkasan Status Kualitas Data (928 Baris)', xlabel='Jumlah baris')
axes[1].invert_yaxis()
fig.tight_layout()
fig.savefig(figure_dir/'06_missing_outlier_status.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print({'iqr_lower': lower, 'iqr_upper': upper, 'outlier_rows': int(df['price_anomaly_iqr'].sum()), 'staging_missing_cells': int(df.isna().sum().sum())})
display(status_counts)

**Interpretasi:** Seluruh **37 sel kosong** pada data mentah telah berhasil dinormalisasi sehingga staging memiliki **0 sel kosong (100% completeness)**. Terdapat **42 kandidat outlier IQR** dan **42 baris terlibat duplikasi logis**, yang dipertahankan melalui kolom flag untuk keperluan audit. Sebanyak **923 baris** dimuat ke `fact_sewa_kos` di warehouse survei dan **833 baris** berstatus `model_ready = 1`.

## Kesimpulan Utama EDA (5 Temuan Kunci)

1. **Konsentrasi Tarif Sewa Bulanan:** Median tarif kos di Kabupaten Toba pada 923 unit valid adalah **Rp700.000/bulan** (rata-rata Rp693.929/bulan, Q1 = Rp575.000, Q3 = Rp800.000).
2. **Disparitas Geografis & Kategori Penghuni:** Kecamatan pusat aktivitas (**Laguboti, Porsea, Ajibata = Rp725.000; Balige = Rp700.000**) memiliki tarif median 1,5–2 kali lipat kecamatan pinggiran (**Pintu Pohan Meranti = Rp352.084; Habinsaran = Rp425.000**). Kos Putri & Campur (Rp725.000) lebih tinggi Rp150.000 dibanding kos Putra (Rp575.000).
3. **Dimensi Fisik Kamar & Lantai:** Luas kamar median adalah **12,0 m²** ($3\times 4$ m), dan unit di **Lantai 2 (median Rp750.000)** memiliki tarif Rp100.000 lebih tinggi dibanding **Lantai 1 (median Rp650.000)**.
4. **Premium Fasilitas & Hierarki Keamanan:** Keberadaan AC (+Rp525.000), Wi-Fi (+Rp200.000), Kamar Mandi Dalam (+Rp200.000), serta tingkat keamanan berjenjang (`Tidak Ada` Rp625.000 $\rightarrow$ `CCTV` Rp650.000 $\rightarrow$ `Penjaga/Satpam` Rp750.000 $\rightarrow$ `Penjaga & CCTV` Rp850.000) menjadi pembeda utama harga.
5. **Pengaruh Aksesibilitas Jarak (Meter):** Jarak masuk dari jalan raya utama (`jarak_jalan_raya_m`) menunjukkan korelasi negatif paling kuat terhadap tarif ($\rho = -0,3975$); unit dalam radius `<= 500 m` dari kampus/kantor/sekolah memiliki median tertinggi (**Rp700.000/bulan**).